In [2]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd


DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="INDICADORES_ESSALUD"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2= engine2.connect()

from datetime import datetime
from sqlalchemy import text

import numpy as np
import oracledb
import pandas as pd
from sqlalchemy import create_engine, text
import sys
from decouple import config

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb

# Configuración de la conexión
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname = config("HOST4_BDI_POSTGRES")
service_name = "WNET"
port = 1527

engine0 = create_engine(f'oracle://{un}:{pw}@', connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()

# Tamaño de los bloques
chunk_size = 500000
offset = 0

# Cargar datos de referencia desde PostgreSQL
tipdoc = pd.read_sql_query("SELECT id_tipdoc, cod_tdo FROM dim_tipdoc", con=engine2)
sexo = pd.read_sql_query("SELECT id_sexo, cod_sex FROM dim_sexo", con=engine2)
estciv = pd.read_sql_query("SELECT id_estciv, cod_ecv FROM dim_estciv", con=engine2)
tipseg = pd.read_sql_query("SELECT id_tipseg, cod_tse FROM dim_tipseg", con=engine2)
plansalud = pd.read_sql_query("SELECT id_plansalud, cod_tse, cod_psa FROM dim_plansalud", con=engine2)
cas = pd.read_sql_query(f"SELECT id_cas,cod_cas, cod_red FROM  dim_cas where cod_cas is not null", con=connection2)
tipoparen = pd.read_sql_query("SELECT id_tippa, cod_tippa FROM dim_tippa", con=engine2)
estreg = pd.read_sql_query("SELECT id_estreg, cod_est FROM dim_estreg", con=engine2)

# Transformaciones y combinaciones con otros DataFrames

def strip_columns(df):
    return df.apply(lambda x: x.str.strip() if x.dtype == "object" and x.apply(type).eq(str).all() else x)


# Aplica la función a cada dataframe
tipdoc = strip_columns(tipdoc)
sexo = strip_columns(sexo)
estciv = strip_columns(estciv)
tipseg = strip_columns(tipseg)
plansalud = strip_columns(plansalud)
cas = strip_columns(cas)
tipoparen = strip_columns(tipoparen)
estreg = strip_columns(estreg)

while True:
    query = f"""
    SELECT *
    FROM (
        SELECT 
            p.PERSECNUM,
            p.PERTIPDOCIDENCOD,
            p.PERDOCIDENNUM,
            p.PERPRINOMDES,
            p.PERSEGNOMDES,
            p.PERAPEPATDES,
            p.PERAPEMATDES,
            p.PERNACFEC,
            p.PERSEXOCOD,
            p.PERESTCIVCOD,
            p.PERTIPSEGCOD,
            p.PERPLANSALUCOD,
            p.PERCENASIADSCOD,
            p.PERINSFEC,
            p.PERVIGFEC,
            p.PERFALFEC,
            p.PERCREFEC,
            p.PERMODFEC,
            p.PERTIPOPARECOD,
            p.PERUBIGEONAC,
            p.PERUBIGEODOM,
            p.PERESTREGCOD,
            ROW_NUMBER() OVER (ORDER BY persecnum) AS rn
        FROM SGSS.CMPER10 p
    )
    WHERE rn > {offset} AND rn <= {offset + chunk_size}
    """

    chunk = pd.read_sql_query(query, con=engine0)
    if chunk.empty:
        break

    chunk = chunk.rename(columns={
        'persecnum': 'per_sec',
        'pertipdocidencod': 'cod_tdo',
        'perdocidennum': 'num_doc',
        'perprinomdes': 'pri_nom',
        'persegnomdes': 'seg_nom',
        'perapepatdes': 'ape_pat',
        'perapematdes': 'ape_mat',
        'pernacfec': 'fec_nac',
        'persexocod': 'cod_sex',
        'perestcivcod': 'cod_ecv',
        'pertipsegcod': 'cod_tse',
        'perplansalucod': 'cod_psa',
        'percenasiadscod': 'cod_cas',
        'perinsfec': 'fec_ins',
        'pervigfec': 'fec_vig',
        'perfalfec': 'fec_fal',
        'percrefec': 'fec_cre',
        'permodfec': 'fec_mod',
        'pertipoparecod': 'cod_tippa',
        'perubigeonac': 'ubi_nac',
        'perubigeodom': 'ubi_dom',
        'perestregcod': 'cod_est'
    })
    
    chunk = strip_columns(chunk)

    chunk = pd.merge(chunk, tipdoc, on='cod_tdo', how='left').drop('cod_tdo', axis=1)

    chunk = pd.merge(chunk, sexo, on='cod_sex', how="left").drop('cod_sex', axis=1)

    chunk = pd.merge(chunk, estciv, on='cod_ecv', how='left').drop('cod_ecv', axis=1)

    chunk = pd.merge(chunk, plansalud, on=['cod_tse', 'cod_psa'], how="left").drop('cod_psa', axis=1)

    chunk = pd.merge(chunk, tipseg, on='cod_tse', how="left").drop('cod_tse', axis=1)

    chunk = pd.merge(chunk, cas,how='left',on='cod_cas').drop('cod_cas', axis=1)

    chunk = pd.merge(chunk, tipoparen, on='cod_tippa', how="left").drop('cod_tippa', axis=1)

    chunk = pd.merge(chunk, estreg, on='cod_est', how="left").drop('cod_est', axis=1)

    # Convertir columnas de fecha a datetime y manejar NaT
    date_columns = ['fec_nac', 'fec_ins', 'fec_vig', 'fec_fal', 'fec_cre', 'fec_mod']

    # Convertir las columnas de fecha a datetime, reemplazando errores con NaT
    for col in date_columns:
        chunk[col] = pd.to_datetime(chunk[col], errors='coerce')

    # Reemplazar NaT por None en columnas de fecha
    for col in date_columns:
        chunk[col] = chunk[col].apply(lambda x: None if pd.isna(x) else x)

    # Convertir el resto de NaN a None
    chunk = chunk.applymap(lambda x: None if pd.isna(x) else x)

    print('subiendo bloques')
    # Upsert en PostgreSQL
    for _, row in chunk.iterrows():
        row_dict = row.to_dict()

        def clean_data(row_dict):
            for key, value in row_dict.items():
                if isinstance(value, str):
                    # Reemplaza o elimina el carácter nulo
                    row_dict[key] = value.replace('\x00', '')
            return row_dict

        # Asumiendo que 'row_dict' es tu diccionario de datos
        row_dict = clean_data(row_dict)

        # Verificar y convertir cualquier valor de cadena 'NaT' o pd.NaT a None
        for key, value in row_dict.items():
            if value == 'NaT' or value is pd.NaT or str(value) == 'NaT':
                row_dict[key] = None


        sql = text("""
        INSERT INTO dim_paciente (per_sec, id_tipdoc, num_doc, pri_nom, seg_nom, ape_pat, ape_mat, fec_nac, id_sexo, id_estciv, id_tipseg, id_cas, fec_ins, fec_vig, fec_fal, fec_cre, fec_mod, id_plansalud, id_tippa, ubi_nac, ubi_dom, id_estreg)
        VALUES (:per_sec, :id_tipdoc, :num_doc, :pri_nom, :seg_nom, :ape_pat, :ape_mat, :fec_nac, :id_sexo, :id_estciv, :id_tipseg, :id_cas, :fec_ins, :fec_vig, :fec_fal, :fec_cre, :fec_mod, :id_plansalud, :id_tippa, :ubi_nac, :ubi_dom, :id_estreg)
        """)
        
        engine2.execute(sql, **row_dict)
    print(f'bloque {_} subido')                                                                                                                                                                                      
    offset += chunk_size

subiendo bloques
bloque 500000 subido
subiendo bloques
bloque 500002 subido
subiendo bloques
bloque 500003 subido
subiendo bloques
bloque 500000 subido
subiendo bloques
bloque 500000 subido
subiendo bloques
bloque 500001 subido
subiendo bloques
bloque 500002 subido
subiendo bloques
bloque 500001 subido
subiendo bloques
bloque 500003 subido
subiendo bloques
bloque 500002 subido
subiendo bloques
bloque 499999 subido
subiendo bloques
bloque 500002 subido
subiendo bloques
bloque 500000 subido
subiendo bloques
bloque 500000 subido
subiendo bloques
bloque 500000 subido
subiendo bloques
bloque 499999 subido
subiendo bloques
bloque 499999 subido
subiendo bloques
bloque 499999 subido
subiendo bloques
bloque 499999 subido
subiendo bloques
bloque 499999 subido
subiendo bloques
bloque 499999 subido
subiendo bloques
bloque 499999 subido
subiendo bloques
bloque 499999 subido
subiendo bloques
bloque 499999 subido
subiendo bloques
bloque 499999 subido
subiendo bloques
bloque 499999 subido
subiendo blo

In [3]:
connection2.close()
connection0.close()
engine2.dispose()
engine0.dispose()